# Model Comparison Analysis\n\nLoads all experiment outputs, verifies test-split alignment, then runs:\n- ROC curves\n- Score distributions (normal vs attack)\n- Pairwise Spearman correlation\n- Simple ensemble\n- Per-model summary table

In [ ]:
import json
from pathlib import Path
from itertools import combinations

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, gaussian_kde
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, precision_score, recall_score

# Locate project root regardless of where Jupyter set the CWD
_cwd = Path.cwd()
if (_cwd / "outputs").exists():
    PROJECT_ROOT = _cwd                  # CWD is already the project root
elif (_cwd.parent / "outputs").exists():
    PROJECT_ROOT = _cwd.parent          # CWD is notebooks/
else:
    # Walk up until we find the outputs dir
    PROJECT_ROOT = next(
        (p for p in [_cwd, *_cwd.parents] if (p / "outputs").exists()),
        _cwd,
    )

OUTPUTS_ROOT = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "report " / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Outputs root : {OUTPUTS_ROOT}  (exists={OUTPUTS_ROOT.exists()})")
print(f"Figures dir  : {FIGURES_DIR}")

## 1. Load experiment results\n\nFor each (model, scenario) group we keep the **most recent** run only.

In [ ]:
def _load_scores_from_dir(run_dir: Path) -> dict | None:
    summary_path = run_dir / "run_summary.json"
    if not summary_path.exists():
        return None
    summary = json.loads(summary_path.read_text())
    model = summary.get("summary", {}).get("model", "unknown")

    metrics_candidates = {
        "gnn":      run_dir / "metrics_gnn.json",
        "tgn":      run_dir / "metrics_tgn.json",
        "pagerank": run_dir / "metrics_pagerank.json",
    }
    metrics_path = metrics_candidates.get(model)
    if metrics_path is None or not metrics_path.exists():
        return None

    metrics = json.loads(metrics_path.read_text())

    # Scores are stored under test_scores (added after compact-metrics refactor)
    scores_block = metrics.get("test_scores", {})
    y_true  = scores_block.get("y_true")
    y_score = scores_block.get("y_score")

    # Pagerank stores them under result.diagnostics in run_summary.json
    if y_true is None or y_score is None:
        diag = summary.get("result", {}).get("diagnostics", {})
        y_true  = diag.get("test_labels")
        y_score = diag.get("test_scores")

    if y_true is None or y_score is None:
        print(f"  [skip] {run_dir.name}: no test scores found in {metrics_path.name}")
        return None

    return {
        "model":    model,
        "scenario": summary.get("summary", {}).get("scenario", "all"),
        "dataset":  summary.get("summary", {}).get("dataset", "unknown"),
        "run_dir":  str(run_dir),
        "y_true":   np.array(y_true, dtype=int),
        "y_score":  np.array(y_score, dtype=float),
        "architecture": summary.get("summary", {}).get("architecture", model),
        "val_f1":   metrics.get("validation", {}).get("f1", float("nan")),
    }


def load_all_runs(outputs_root: Path) -> list[dict]:
    runs = []
    for summary_path in sorted(outputs_root.rglob("run_summary.json")):
        result = _load_scores_from_dir(summary_path.parent)
        if result is not None:
            runs.append(result)
    return runs


def latest_per_group(runs: list[dict]) -> dict[tuple, dict]:
    groups: dict[tuple, dict] = {}
    for run in runs:
        key = (run["model"], run["scenario"], run["dataset"])
        groups[key] = run  # sorted by path/timestamp, last wins
    return groups


all_runs  = load_all_runs(OUTPUTS_ROOT)
best_runs = latest_per_group(all_runs)

print(f"\nLoaded {len(best_runs)} runs:\n")
for key, run in sorted(best_runs.items()):
    auc = roc_auc_score(run["y_true"], run["y_score"])
    print(f"  {run['model']:10s}  scenario={run['scenario']:25s}  "
          f"n_test={len(run['y_true'])}  AUC={auc:.4f}  val_f1={run['val_f1']:.4f}")

## 2. Alignment check\n\nCorrelation is only meaningful if models were evaluated on the **same test samples in the same order**. We verify by comparing `y_true` sequences.

In [ ]:
# Group runs by (scenario, dataset) — only compare within the same group
from collections import defaultdict

groups_for_comparison: dict[tuple, list[dict]] = defaultdict(list)
for run in best_runs.values():
    groups_for_comparison[(run["scenario"], run["dataset"])].append(run)

for group_key, group_runs in sorted(groups_for_comparison.items()):
    print(f"\n--- scenario={group_key[0]}  dataset={group_key[1]} ---")
    if len(group_runs) < 2:
        print("  Only one model — nothing to compare.")
        continue
    ref = group_runs[0]
    for run in group_runs[1:]:
        if len(run["y_true"]) != len(ref["y_true"]):
            print(f"  ⚠️  {run['model']} vs {ref['model']}: DIFFERENT test-set sizes "
                  f"({len(run['y_true'])} vs {len(ref['y_true'])}) — cannot compare directly.")
        elif not np.array_equal(run["y_true"], ref["y_true"]):
            print(f"  ⚠️  {run['model']} vs {ref['model']}: same size but y_true differs — "
                  f"different splits, scores are NOT aligned.")
        else:
            print(f"  ✓  {run['model']} and {ref['model']} share the same test split.")

## 3. ROC curves

In [ ]:
COLORS = {"gnn": "#2196F3", "tgn": "#FF5722", "pagerank": "#4CAF50"}
LABELS = {"gnn": "GCN (static)", "tgn": "TGN (temporal)", "pagerank": "PageRank"}

for group_key, group_runs in sorted(groups_for_comparison.items()):
    scenario, dataset = group_key
    if len(group_runs) < 2:
        continue

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Random")

    for run in sorted(group_runs, key=lambda r: r["model"]):
        fpr, tpr, _ = roc_curve(run["y_true"], run["y_score"])
        auc = roc_auc_score(run["y_true"], run["y_score"])
        label = LABELS.get(run["model"], run["model"])
        ax.plot(fpr, tpr, lw=2, color=COLORS.get(run["model"], "gray"),
                label=f"{label}  (AUC={auc:.3f})")

    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC — {dataset} / {scenario}")
    ax.legend(fontsize=9)
    fig.tight_layout()
    fname = FIGURES_DIR / f"roc_{dataset}_{scenario}.pdf"
    fig.savefig(fname)
    print(f"Saved {fname}")
    plt.show()

## 4. Score distributions\n\nKDE of predicted attack-probability scores, split by true label. Shows whether each model's scores separate the classes cleanly.

In [ ]:
for group_key, group_runs in sorted(groups_for_comparison.items()):
    scenario, dataset = group_key
    n_models = len(group_runs)
    if n_models == 0:
        continue

    fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 3.5))
    if n_models == 1:
        axes = [axes]

    for ax, run in zip(axes, sorted(group_runs, key=lambda r: r["model"])):
        scores = run["y_score"]
        labels = run["y_true"]

        # Adaptive x-axis: zoom into where the actual scores live
        lo = max(0.0, float(np.percentile(scores, 1))  - 0.02)
        hi = min(1.0, float(np.percentile(scores, 99)) + 0.02)
        xs = np.linspace(lo, hi, 300)

        for cls, color, name in [(0, "#2196F3", "Normal"), (1, "#FF5722", "Attack")]:
            s = scores[labels == cls]
            if len(s) < 2:
                continue
            kde = gaussian_kde(s, bw_method=0.15)
            ys  = np.clip(kde(xs), 0, None)   # no negative density
            ax.fill_between(xs, ys, alpha=0.35, color=color, label=name)
            ax.plot(xs, ys, color=color, lw=1.5)

        ax.axvline(0.5, color="k", lw=0.8, ls="--", alpha=0.4, label="thr=0.5")
        ax.set_title(LABELS.get(run["model"], run["model"]), fontsize=10)
        ax.set_xlabel("Predicted attack score")
        ax.set_ylabel("Density")
        ax.legend(fontsize=8)

        # Print score stats so calibration issues are immediately visible
        for cls, name in [(0, "Normal"), (1, "Attack")]:
            s = scores[labels == cls]
            print(f"  {LABELS.get(run['model'],run['model']):<20} {name}:  "
                  f"mean={s.mean():.4f}  std={s.std():.4f}  "
                  f"min={s.min():.4f}  max={s.max():.4f}")

    fig.suptitle(f"Score distributions — {dataset} / {scenario}", fontsize=11)
    fig.tight_layout()
    fname = FIGURES_DIR / f"score_dist_{dataset}_{scenario}.pdf"
    fig.savefig(fname)
    plt.show()

## 5. Pairwise score correlation\n\nSpearman ρ between model scores on the same test set. High ρ → models capture the same signal. Low ρ → complementary — ensembling would help.

In [ ]:
for group_key, group_runs in sorted(groups_for_comparison.items()):
    scenario, dataset = group_key
    # Only compare runs that share the exact same test split
    aligned = [r for r in group_runs
               if np.array_equal(r["y_true"], group_runs[0]["y_true"])
               and len(r["y_true"]) == len(group_runs[0]["y_true"])]
    if len(aligned) < 2:
        continue

    names  = [LABELS.get(r["model"], r["model"]) for r in aligned]
    scores = [r["y_score"] for r in aligned]
    n = len(aligned)
    corr_matrix = np.eye(n)
    for i, j in combinations(range(n), 2):
        rho, _ = spearmanr(scores[i], scores[j])
        corr_matrix[i, j] = corr_matrix[j, i] = rho

    fig, ax = plt.subplots(figsize=(3 + n, 2.5 + n * 0.4))
    im = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap="RdYlGn")
    ax.set_xticks(range(n)); ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_yticks(range(n)); ax.set_yticklabels(names)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{corr_matrix[i, j]:.2f}", ha="center", va="center",
                    fontsize=11, color="black")
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(f"Spearman ρ — {dataset} / {scenario}")
    fig.tight_layout()
    fname = FIGURES_DIR / f"score_correlation_{dataset}_{scenario}.pdf"
    fig.savefig(fname)
    print(f"Saved {fname}")
    plt.show()

    print(f"\nPairwise Spearman ρ  ({dataset} / {scenario})")
    for i, j in combinations(range(n), 2):
        print(f"  {names[i]:20s} vs {names[j]:20s}  ρ={corr_matrix[i,j]:.3f}")

## 6. Ensemble\n\nSimple score average over aligned models. Worthwhile only if correlation is low.

In [ ]:
def best_f1_threshold(y_true, y_score):
    """Sweep thresholds and return the one maximising F1."""
    unique = sorted(set(y_score))
    candidates = [(a + b) / 2 for a, b in zip(unique, unique[1:])] or [0.5]
    best_thr, best_f1 = 0.5, 0.0
    for thr in candidates:
        f1 = f1_score(y_true, (y_score >= thr).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return best_thr, best_f1


for group_key, group_runs in sorted(groups_for_comparison.items()):
    scenario, dataset = group_key
    aligned = [r for r in group_runs
               if np.array_equal(r["y_true"], group_runs[0]["y_true"])
               and len(r["y_true"]) == len(group_runs[0]["y_true"])]
    if len(aligned) < 2:
        continue

    y_true = aligned[0]["y_true"]
    print(f"\n{'='*60}")
    print(f"{dataset} / {scenario}")
    print(f"{'Model':<22} {'AUC':>7} {'F1':>7} {'Prec':>7} {'Rec':>7}  thr")

    individual_scores = []
    for run in sorted(aligned, key=lambda r: r["model"]):
        s = run["y_score"]
        individual_scores.append(s)
        auc = roc_auc_score(y_true, s)
        thr, f1 = best_f1_threshold(y_true, s)
        y_pred = (s >= thr).astype(int)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec  = recall_score(y_true, y_pred, zero_division=0)
        name = LABELS.get(run["model"], run["model"])
        print(f"  {name:<20} {auc:>7.4f} {f1:>7.4f} {prec:>7.4f} {rec:>7.4f}  {thr:.3f}")

    # Ensemble
    ens = np.mean(individual_scores, axis=0)
    auc = roc_auc_score(y_true, ens)
    thr, f1 = best_f1_threshold(y_true, ens)
    y_pred = (ens >= thr).astype(int)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    print(f"  {'Ensemble (avg)':<20} {auc:>7.4f} {f1:>7.4f} {prec:>7.4f} {rec:>7.4f}  {thr:.3f}")